## Step 1: Data Cleaning & Feature Engineering

In [2]:
import pandas as pd
import numpy as np
import re

# Load raw dataset
df_raw = pd.read_csv('pakistan-startup-census.csv')

In [3]:
# 2. Filter out completely null rows (120 blank records)
df_cleaned = df_raw.dropna(how='all').copy()
df_cleaned

,Name,Location,Tagline,Category,Website,Founded,Description
0,Outnet,Karachi Pakistan,Cloud based SaaS platform for planning procur...,Advertising OOH Outdoor Advertising Data an...,http://www.outnet.com.pk,1st September 2014,24Grey is a T4M (technology for marketing) com...
1,7Vals,Lahore Pakistan,Understanding and enabling businesses to impro...,Consulting Rails Product Development UI,http://www.7vals.com,2011,We're changing the way businesses work by maki...
2,92Solution,Lahore Pakistan,92 Solution A Pakistani Company Giving his Pa...,Consulting Website & Software Development,http://92solution.com,7th January 2015,92 Solution is a Software & Website Developmen...
3,A2Z Yellow Pages & Info Services,Lahore,Find local Businesses and Services in Pakistan...,Online Business Directory & Portal,https://www.yp-pak.com,30th January 2017,A2Z| Yellow Pages Pakistan will be the innovat...
4,AALogics,Karachi Pakistan,NaN,Consulting Software,http://www.aalogics.com,1st August 2014,IT Consulting Web development & Designing Lo...
...,...,...,...,...,...,...,...
428,Yum to Yikes,Islamabad Pakistan,Yum to Yikes aims to inform entertain and gui...,Media/News,http://yumtoyikes.com,1st May 2015,We are a bunch of enthusiastically-creative fo...
429,Zambeel,Multan Pakistan,Online Shopping for Quality Products at lowest...,E-commerce,https://www.zambeel.com,2015,Online Shopping for Quality Products at lowest...
430,Zameen,Lahore Pakistan,Zameen.com has single-handedly revolutionized ...,Property Portal,http://www.zameen.com/,2006,NaN
431,Zest,Islamabad Pakistan,Zest is the Pakistan's first on-demand massage...,Health,NaN,10th December 2016,Zest is the Massage On Demand company. Availab...


In [4]:
# 3. Standardize Locations into a clean City column
def extract_city(loc):
    if pd.isna(loc):
        return 'Unknown'
    loc_str = str(loc).strip()
    loc_lower = loc_str.lower()
    
    if 'karachi' in loc_lower:
        return 'Karachi'
    elif 'lahore' in loc_lower:
        return 'Lahore'
    elif 'islamabad' in loc_lower:
        return 'Islamabad'
    elif 'rawalpindi' in loc_lower:
        return 'Rawalpindi'
    elif 'peshawar' in loc_lower:
        return 'Peshawar'
    elif 'faisalabad' in loc_lower:
        return 'Faisalabad'
    elif 'multan' in loc_lower:
        return 'Multan'
    elif 'quetta' in loc_lower:
        return 'Quetta'
    elif 'sialkot' in loc_lower:
        return 'Sialkot'
    elif any(term in loc_lower for term in ['san francisco', 'california', 'usa', 'united states']):
        return 'International / US'
    elif loc_lower in ['pakistan', '']:
        return 'Unspecified Pakistan'
    else:
        return loc_str

df_cleaned['Clean_City'] = df_cleaned['Location'].apply(extract_city)
df_cleaned['Clean_City']

0        Karachi
1         Lahore
2         Lahore
3         Lahore
4        Karachi
         ...    
428    Islamabad
429       Multan
430       Lahore
431    Islamabad
432       Lahore
Name: Clean_City, Length: 433, dtype: object

In [5]:
# 4. Parse Founding Dates into Founding_Year and Founding_Month
def extract_founding_year(val):
    if pd.isna(val):
        return None
    match = re.search(r'\b(19\d\d|20\d\d)\b', str(val))
    return int(match.group(1)) if match else None

months_list = ['january', 'february', 'march', 'april', 'may', 'june', 
               'july', 'august', 'september', 'october', 'november', 'december']

def extract_founding_month(val):
    if pd.isna(val):
        return None
    val_lower = str(val).lower()
    for month in months_list:
        if month in val_lower:
            return month.capitalize()
    return None

df_cleaned['Founding_Year'] = df_cleaned['Founded'].apply(extract_founding_year)
df_cleaned['Founding_Month'] = df_cleaned['Founded'].apply(extract_founding_month)

df_cleaned['Founding_Year']
df_cleaned['Founding_Month']

0      September
1           None
2        January
3        January
4         August
         ...    
428          May
429         None
430         None
431     December
432      January
Name: Founding_Month, Length: 433, dtype: object

In [6]:
# 5. Extract Top-Level Domains (TLD)
def extract_tld(url):
    if pd.isna(url) or not str(url).strip():
        return 'Missing'
    url_str = str(url).lower().strip()
    if '.com.pk' in url_str:
        return '.com.pk'
    elif '.pk' in url_str:
        return '.pk'
    elif '.com' in url_str:
        return '.com'
    elif '.co' in url_str:
        return '.co'
    elif '.io' in url_str:
        return '.io'
    elif '.org' in url_str:
        return '.org'
    elif '.net' in url_str:
        return '.net'
    else:
        return 'Other'

df_cleaned['Domain_Extension'] = df_cleaned['Website'].apply(extract_tld)

df_cleaned['Domain_Extension']

0      .com.pk
1         .com
2         .com
3         .com
4         .com
        ...   
428       .com
429       .com
430       .com
431    Missing
432        .pk
Name: Domain_Extension, Length: 433, dtype: object

In [7]:
# 6. Save cleaned dataset for subsequent project steps
df_cleaned.to_csv('pakistan_startup_census_cleaned.csv', index=False)

## Step 2: Feature Engineering Implementation

In [8]:
# Load the cleaned dataset from Step 1
df = pd.read_csv('pakistan_startup_census_cleaned.csv')

In [9]:
# 1. Primary Category Standardization

def classify_category(cat):
    if pd.isna(cat):
        return 'Other / General'
    
    cat_lower = str(cat).lower()
    
    # Mapping raw sub-industries into standardized high-level sectors
    if any(k in cat_lower for k in ['e-commerce', 'ecommerce', 'retail', 'marketplace', 'shopping', 'store', 'fashion', 'clothing']):
        return 'E-Commerce & Retail'
    elif any(k in cat_lower for k in ['education', 'edtech', 'school', 'learning', 'training', 'e-learning', 'academy']):
        return 'EdTech & Education'
    elif any(k in cat_lower for k in ['health', 'healthcare', 'medical', 'hospital', 'doctor', 'pharma', 'fitness']):
        return 'HealthTech & Wellness'
    elif any(k in cat_lower for k in ['fintech', 'finance', 'financial', 'banking', 'payment', 'insurance', 'crypto', 'accounting']):
        return 'FinTech & Financial Services'
    elif any(k in cat_lower for k in ['advertising', 'marketing', 'media', 'seo', 'agency', 'pr', 'creative', 'design']):
        return 'Digital Marketing & Media'
    elif any(k in cat_lower for k in ['consulting', 'services', 'bpo', 'outsourcing', 'recruitment', 'hr', 'jobs', 'staffing']):
        return 'IT Consulting & Services'
    elif any(k in cat_lower for k in ['saas', 'software', 'technology', 'web development', 'app development', 'data', 'analytics', 'cloud', 'ai', 'mobile']):
        return 'Software & SaaS'
    else:
        return 'Other / General'

df['Primary_Category'] = df['Category'].apply(classify_category)


In [10]:
# 2. Startup Age Calculation

REFERENCE_YEAR = 2026
df['Startup_Age'] = df['Founding_Year'].apply(lambda y: int(REFERENCE_YEAR - y) if pd.notna(y) else None)
df['Startup_Age'] 

0      12.0
1      15.0
2      11.0
3       9.0
4      12.0
       ... 
428    11.0
429    11.0
430    20.0
431    10.0
432    10.0
Name: Startup_Age, Length: 433, dtype: float64

In [11]:
# 3. Completeness Flags (Has_Website & Has_Description)

df['Has_Website'] = df['Website'].apply(lambda x: 1 if pd.notna(x) and str(x).strip() != '' else 0)
df['Has_Description'] = df['Description'].apply(lambda x: 1 if pd.notna(x) and str(x).strip() != '' else 0)

In [12]:
# 4. Export Featured Dataset

df.to_csv('pakistan_startup_census_featured.csv', index=False)

## Step 3: Exploratory Data Analysis

In [13]:
# Load the featured dataset from Step 2
df = pd.read_csv('pakistan_startup_census_featured.csv')

In [14]:
# Q1: City Share Analysis
city_counts = df['Clean_City'].value_counts()
city_pct = df['Clean_City'].value_counts(normalize=True) * 100
city_summary = pd.DataFrame({'Count': city_counts, 'Percentage (%)': city_pct.round(2)})
print("--- City Distribution ---")
print(city_summary.head(10))

--- City Distribution ---
                      Count  Percentage (%)
Clean_City                                 
Karachi                 200           46.19
Lahore                  122           28.18
Islamabad                53           12.24
International / US       10            2.31
Faisalabad                5            1.15
Peshawar                  5            1.15
Unspecified Pakistan      4            0.92
Rawalpindi                4            0.92
Sialkot                   2            0.46
Dubai                     2            0.46


In [15]:
# Q2: Peak Growth Years
year_counts = df['Founding_Year'].value_counts().sort_index()
print("\n--- Founding Year Distribution ---")
print(year_counts)


--- Founding Year Distribution ---
Founding_Year
1957.0      1
1984.0      1
1997.0      1
1999.0      1
2000.0      2
2001.0      1
2002.0      1
2003.0      1
2004.0      3
2005.0      4
2006.0      3
2007.0      1
2008.0      4
2009.0      2
2010.0      7
2011.0      6
2012.0     11
2013.0     16
2014.0     44
2015.0     91
2016.0    104
2017.0     46
2018.0      7
Name: count, dtype: int64


In [17]:
# Q3: Domain Extension Analysis (.pk/.com.pk vs .com)
domain_counts = df['Domain_Extension'].value_counts()
domain_pct = df['Domain_Extension'].value_counts(normalize=True) * 100
domain_summary = pd.DataFrame({'Count': domain_counts, 'Percentage (%)': domain_pct.round(2)})
print("\n--- Domain Extension Breakdown ---")
print(domain_summary)

local_tld_pct = df['Domain_Extension'].isin(['.pk', '.com.pk']).mean() * 100
com_tld_pct = (df['Domain_Extension'] == '.com').mean() * 100
print(f"\nLocal TLD (.pk / .com.pk) Share: {local_tld_pct:.2f}%")
print(f"International TLD (.com) Share: {com_tld_pct:.2f}%")


--- Domain Extension Breakdown ---
                  Count  Percentage (%)
Domain_Extension                       
.com                216           49.88
.pk                 127           29.33
.co                  22            5.08
.com.pk              18            4.16
Missing              15            3.46
.io                  13            3.00
Other                10            2.31
.net                  8            1.85
.org                  4            0.92

Local TLD (.pk / .com.pk) Share: 33.49%
International TLD (.com) Share: 49.88%


In [18]:
# Q4: Top 5 Saturated Categories in Karachi vs Lahore vs Islamabad
top_cities = ['Karachi', 'Lahore', 'Islamabad']
for city in top_cities:
    print(f"\n--- Top 5 Categories in {city} ---")
    sub_df = df[df['Clean_City'] == city]
    print(sub_df['Primary_Category'].value_counts().head(5))


--- Top 5 Categories in Karachi ---
Primary_Category
Other / General              40
E-Commerce & Retail          39
Software & SaaS              37
IT Consulting & Services     24
Digital Marketing & Media    22
Name: count, dtype: int64

--- Top 5 Categories in Lahore ---
Primary_Category
Other / General             32
E-Commerce & Retail         26
IT Consulting & Services    18
Software & SaaS             16
HealthTech & Wellness       11
Name: count, dtype: int64

--- Top 5 Categories in Islamabad ---
Primary_Category
Other / General              18
Software & SaaS               9
IT Consulting & Services      7
E-Commerce & Retail           6
Digital Marketing & Media     6
Name: count, dtype: int64
